## Apply UMAP to CRISPR perturbations

These are the Cell Painting profiles of the CRISPR perturbations used to train and test each Cell Health model

In [ ]:
import umap
import pathlib
import numpy as np
import pandas as pd

In [7]:
np.random.seed(123)

In [8]:
# Set constants and file names
consensus = "modz"

data_dir = pathlib.Path("..", "1.generate-profiles", "data", "consensus")
cell_process_dir = pathlib.Path("..", "1.generate-profiles", "tables")
results_dir = pathlib.Path("results")
shiny_app_dir = pathlib.Path("..", "4.apply", "repurposing_cellhealth_shiny", "data")

profile_file = pathlib.Path(data_dir, f"cell_painting_{consensus}.tsv.gz")
cell_health_file = pathlib.Path(data_dir, f"cell_health_{consensus}.tsv.gz")

cell_process_file = pathlib.Path(cell_process_dir, "supplementary_table_1_perturbation_details.tsv")
output_file = pathlib.Path(shiny_app_dir, f"profile_umap_with_cell_health_{consensus}.tsv")

In [ ]:
# Load profile data
df = (
    pd.read_csv(profile_file, sep="\t")
    .sort_values(by="Metadata_profile_id")
    .reset_index(drop=True)
)

# Remove the problematic level_0 column if it exists
if 'level_0' in df.columns:
    df = df.drop('level_0', axis=1)

# Manually identify CellProfiler features instead of using infer_cp_features
# which is failing with the current pycytominer version
metadata_cols = [col for col in df.columns if col.startswith('Metadata_')]
cp_features = [col for col in df.columns if col.startswith(('Cells_', 'Nuclei_', 'Cytoplasm_', 'Image_'))]

print(f"Data shape: {df.shape}")
print(f"Metadata columns: {len(metadata_cols)}")
print(f"CellProfiler features: {len(cp_features)}")

print(df.shape)
df.head()
# VALIDATION: Check if we have sufficient data
if df.empty:
    print("ERROR: Profile data is empty. Cannot proceed with UMAP.")
    exit(1)

if len(cp_features) < 10:
    print("ERROR: Insufficient CellProfiler features for UMAP. Need at least 10 features.")
    exit(1)

print(f"Data validation passed: {len(cp_features)} features, {df.shape[0]} profiles")


Data shape: (119, 410)
Metadata columns: 3
CellProfiler features: 407
(119, 410)


,Metadata_profile_id,Metadata_cell_line,Metadata_pert_name,Image_Count_Cells,Image_Count_Cytoplasm,Image_Count_Nuclei,Image_ExecutionTime_01LoadData,Image_ExecutionTime_02CorrectIlluminationApply,Image_ExecutionTime_03MeasureImageQuality,Image_ExecutionTime_04MeasureImageQuality,...,Image_Texture_Variance_RNA_20_0,Image_Texture_Variance_RNA_5_0,Image_Threshold_FinalThreshold_Cells,Image_Threshold_FinalThreshold_Nuclei,Image_Threshold_OrigThreshold_Cells,Image_Threshold_OrigThreshold_Nuclei,Image_Threshold_SumOfEntropies_Cells,Image_Threshold_SumOfEntropies_Nuclei,Image_Threshold_WeightedVariance_Cells,Image_Threshold_WeightedVariance_Nuclei
0,profile_0,A549,AKT1-1,-1.217374,-1.217374,-1.217374,-6.220304,-5.733171,-15.102998,-9.892531,...,-0.527583,-0.527812,-1.835659,-0.082836,-1.835659,-0.082836,-1.545126,0.237515,1.878963,0.121606
1,profile_1,A549,AKT1-2,-0.444177,-0.444177,-0.444177,-0.749434,-2.360718,-9.275025,-7.194568,...,0.095836,0.120245,-1.177413,-0.245552,-1.177413,-0.245552,-0.510188,-0.405942,1.303887,0.155208
2,profile_10,A549,BCL2-2,-1.365433,-1.365433,-1.365433,0.412189,-0.337245,0.261093,0.562076,...,-0.109501,-0.114330,-0.693208,-0.047353,-0.693208,-0.047353,-1.218166,-0.033734,1.043968,-0.596763
3,profile_100,A549,RAF1-2,-1.505266,-1.505266,-1.505266,0.618283,0.337245,-0.155413,1.573812,...,-0.778609,-0.785057,-0.683914,-0.494962,-0.683914,-0.494962,-1.000569,0.398137,0.900962,-0.945581
4,profile_101,A549,RHOA-1,-0.904805,-0.904805,-0.904805,1.498868,-0.674491,-1.150053,-0.674491,...,-0.511344,-0.521686,-1.164343,-0.502437,-1.164343,-0.502437,-0.851858,1.287121,0.852570,-1.026407


In [10]:
# Load cell health data
cell_health_df = (
    pd.read_csv(cell_health_file, sep="\t")
    .sort_values(by="Metadata_profile_id")
    .reset_index(drop=True)
)

print(cell_health_df.shape)
cell_health_df.head()

(119, 76)


,index,cell_id,guide,cc_all_high_h2ax,cc_all_large_notround_polynuclear_mean,cc_all_large_round_polyploid_mean,cc_all_n_objects,cc_all_n_spots_h2ax_mean,cc_all_n_spots_h2ax_per_nucleus_area_mean,cc_all_nucleus_area_mean,...,vb_percent_dead,vb_percent_dead_only,vb_percent_early_apoptosis,vb_percent_late_apoptosis,vb_percent_live,vb_ros_back_mean,vb_ros_mean,Metadata_profile_id,Metadata_pert_name,Metadata_cell_line
0,0,A549,AKT1-1,-0.005795,0.580351,0.013975,0.381958,0.150696,0.162511,-0.167603,...,-0.020236,-0.007970,0.082424,0.000000,0.020263,0.408214,0.654575,profile_0,AKT1-1,A549
1,1,A549,AKT1-2,0.050169,1.277730,0.241808,0.577422,0.220829,0.366989,-0.278044,...,0.225091,0.220461,0.132834,0.386327,-0.224965,0.284962,0.567898,profile_1,AKT1-2,A549
2,10,A549,BCL2-2,-0.182172,0.270253,-0.165335,1.022081,-0.443078,-0.456076,-0.595775,...,-0.015343,-0.013721,0.000000,0.118264,0.015366,0.232668,1.311557,profile_10,BCL2-2,A549
3,102,A549,RAF1-2,0.064110,0.025085,0.014670,-0.231155,0.053950,0.058550,0.459818,...,-0.013249,-0.016064,0.081094,0.163820,0.013285,-1.057565,-0.888677,profile_100,RAF1-2,A549
4,103,A549,RHOA-1,1.767355,1.501962,0.626605,-0.328481,1.599174,2.183969,0.467122,...,-0.205431,-0.204576,0.083873,0.000000,0.205403,-0.574627,-0.314264,profile_101,RHOA-1,A549


In [11]:
# Load cell process annotation file
cell_process_df = pd.read_csv(cell_process_file, sep="\t")
cell_process_df.columns = [f"Metadata_{x}" for x in cell_process_df.columns]

print(cell_process_df.shape)
cell_process_df.head()

(127, 9)


,Metadata_gene_name,Metadata_pert_name,Metadata_broad_sample,Metadata_cell_process,Metadata_cell_health_data,Metadata_cell_painting_data,Metadata_A549_ctg_efficiency,Metadata_ES2_ctg_efficiency,Metadata_HCC44_ctg_efficiency
0,AKT1,AKT1-1,BRDN0001054908,PIK3CA,True,True,0.919690,1.057227,1.022670
1,AKT1,AKT1-2,BRDN0001055115,PIK3CA,True,True,0.842040,0.957005,0.930025
2,ARID1B,ARID1B-1,NaN,Chromatin Modifiers,True,True,0.947521,1.059419,1.026531
3,ARID1B,ARID1B-2,NaN,Chromatin Modifiers,True,True,1.072952,0.970596,0.965475
4,ATF4,ATF4-1,NaN,ER Stress/UPR,True,True,0.964266,1.056071,0.853681


In [12]:
# Ensure data and predictions are aligned
assert df.Metadata_profile_id.tolist() == cell_health_df.Metadata_profile_id.tolist()

In [13]:
# Apply UMAP
reducer = umap.UMAP(random_state=1234, n_components=2)

predict_embedding_df = pd.DataFrame(
    reducer.fit_transform(df.loc[:, cp_features]),
    columns=["umap_x", "umap_y"]
)

print(predict_embedding_df.shape)
predict_embedding_df.head()

/opt/anaconda3/envs/cell-health/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


(119, 2)


,umap_x,umap_y
0,5.825033,3.738619
1,5.372091,3.592877
2,6.246626,4.871582
3,6.496037,4.757114
4,7.250768,4.632611


In [14]:
# Combine data to form a single output file
output_df = (
    predict_embedding_df
    .merge(
        cell_health_df,
        left_index=True,
        right_index=True
    )
    .merge(
        cell_process_df,
        left_on="Metadata_pert_name",
        right_on="Metadata_pert_name",
        how="left"
    )
    # Drops 3 redundant "Empty" pert IDs
    .drop_duplicates(subset="Metadata_profile_id")
)

print(output_df.shape)
output_df.head()

(119, 86)


,umap_x,umap_y,index,cell_id,guide,cc_all_high_h2ax,cc_all_large_notround_polynuclear_mean,cc_all_large_round_polyploid_mean,cc_all_n_objects,cc_all_n_spots_h2ax_mean,...,Metadata_pert_name,Metadata_cell_line,Metadata_gene_name,Metadata_broad_sample,Metadata_cell_process,Metadata_cell_health_data,Metadata_cell_painting_data,Metadata_A549_ctg_efficiency,Metadata_ES2_ctg_efficiency,Metadata_HCC44_ctg_efficiency
0,5.825033,3.738619,0,A549,AKT1-1,-0.005795,0.580351,0.013975,0.381958,0.150696,...,AKT1-1,A549,AKT1,BRDN0001054908,PIK3CA,True,True,0.919690,1.057227,1.022670
1,5.372091,3.592877,1,A549,AKT1-2,0.050169,1.277730,0.241808,0.577422,0.220829,...,AKT1-2,A549,AKT1,BRDN0001055115,PIK3CA,True,True,0.842040,0.957005,0.930025
2,6.246626,4.871582,10,A549,BCL2-2,-0.182172,0.270253,-0.165335,1.022081,-0.443078,...,BCL2-2,A549,BCL2,NaN,Apoptosis,True,True,0.845307,1.137457,1.160181
3,6.496037,4.757114,102,A549,RAF1-2,0.064110,0.025085,0.014670,-0.231155,0.053950,...,RAF1-2,A549,RAF1,NaN,MAPK,True,True,0.940054,1.019956,0.894794
4,7.250768,4.632611,103,A549,RHOA-1,1.767355,1.501962,0.626605,-0.328481,1.599174,...,RHOA-1,A549,RHOA,BRDN0000990371,Cytoskeletal Re-org/Integrin,True,True,0.968079,1.023110,1.092659


In [15]:
# Output to file
output_df.to_csv(output_file, sep="\t", index=False)